# PhenoPred — Dataset Exploration and Structural Validation

**Notebook:** `01_dataset_exploration.ipynb`

**Purpose:** Initial, purely exploratory data analysis (EDA) and structural validation of two raw consumer SNP microarray genotype files belonging to the same individual, prior to any pipeline design.

**Datasets:**
- **Dataset A** — AncestryDNA raw export (`AncestryDNA.txt`)
- **Dataset B** — 23andMe raw export, Build 37 (`anonymous_genome_v5_build37.txt`)

**Scope of this notebook (strict):**
- This notebook performs **structural inspection, profiling, and data-quality observation only**.
- It does **not** clean, normalize, transform, strand-correct, or otherwise modify the underlying data.
- It does **not** compare the two datasets against one another (e.g. no concordance/overlap analysis).
- It does **not** draw any biological, medical, or ancestry-related conclusions.
- All findings are descriptive and intended to inform *future* pipeline design decisions, which are out of scope here.

---

## 1. Setup & Configuration

In [31]:
# Standard library only — no preprocessing libraries are imported,
# since this notebook does not clean, normalize, or transform data.
import os
import io
import csv
import sys
import collections
import platform
from datetime import datetime

print(f"Python version : {sys.version}")
print(f"Platform       : {platform.platform()}")
print(f"Notebook run   : {datetime.now().isoformat()}")

Python version : 3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:13:34) [MSC v.1944 64 bit (AMD64)]
Platform       : Windows-11-10.0.26200-SP0
Notebook run   : 2026-07-07T00:27:42.098977


In [32]:
# ---------------------------------------------------------------------------
# File paths (as provided). No files are moved, renamed, or modified.
# ---------------------------------------------------------------------------
DATASET_A_PATH = r"F:\SRH Munich\2nd sem\Gemonics Analytics\PhenoPred\data\raw\AncestryDNA.txt"                    # Dataset A: AncestryDNA
DATASET_B_PATH = r"F:\SRH Munich\2nd sem\Gemonics Analytics\PhenoPred\data\raw\anonymous_genome_v5_build37.txt"     # Dataset B: 23andMe (Build 37)

DATASET_PATHS = {
    "Dataset A (AncestryDNA)": DATASET_A_PATH,
    "Dataset B (23andMe, Build 37)": DATASET_B_PATH,
}

for label, path in DATASET_PATHS.items():
    exists = os.path.isfile(path)
    size_mb = os.path.getsize(path) / (1024 ** 2) if exists else float("nan")
    print(f"{label:35s} | path='{path}' | exists={exists} | size={size_mb:.2f} MB")

Dataset A (AncestryDNA)             | path='F:\SRH Munich\2nd sem\Gemonics Analytics\PhenoPred\data\raw\AncestryDNA.txt' | exists=True | size=17.47 MB
Dataset B (23andMe, Build 37)       | path='F:\SRH Munich\2nd sem\Gemonics Analytics\PhenoPred\data\raw\anonymous_genome_v5_build37.txt' | exists=True | size=15.26 MB


## 2. Data Loading (No Modifications)

Both files are raw, plain-text, delimited exports from consumer DNA testing services. They are **not** whole-genome sequence files (e.g. not VCF/FASTQ) — they are SNP microarray genotype call lists.

Because both files can be large (several hundred thousand lines), loading is done carefully:
- Files are read as **raw text first** (line-by-line), so we do not assume a delimiter, encoding, or schema before verifying it.
- No rows are dropped, reordered, deduplicated, or altered at this stage.
- Comment/metadata header lines (lines beginning with `#`) are kept separately from data lines for inspection — they are not discarded from the analysis, only separated for clarity.

In [33]:
def load_raw_lines(path, encoding="utf-8", errors="replace"):
    """
    Load a raw text file line-by-line without any modification.
    Returns the list of raw lines (newline characters stripped for display only;
    the underlying file on disk is never touched or rewritten).
    """
    with open(path, "r", encoding=encoding, errors=errors, newline="") as f:
        lines = f.readlines()
    return lines

# Load both datasets as raw lines. Encoding is provisionally set to utf-8 with
# 'replace' error handling purely so the file can be read for inspection;
# the true encoding is verified in Section 3 and not assumed.
raw_lines_a = load_raw_lines(DATASET_A_PATH)
raw_lines_b = load_raw_lines(DATASET_B_PATH)

print(f"Dataset A: {len(raw_lines_a):,} raw lines loaded (including comments/header).")
print(f"Dataset B: {len(raw_lines_b):,} raw lines loaded (including comments/header).")

Dataset A: 677,455 raw lines loaded (including comments/header).
Dataset B: 631,477 raw lines loaded (including comments/header).


In [34]:
def split_comments_and_data(raw_lines, comment_prefix="#"):
    """
    Separate leading/interspersed comment lines from data lines, without
    modifying either. Returns (comment_lines, data_lines), each with original
    line content preserved (trailing newline stripped only for display).
    """
    comment_lines = []
    data_lines = []
    for line in raw_lines:
        stripped = line.rstrip("\r\n")
        if stripped.startswith(comment_prefix):
            comment_lines.append(stripped)
        else:
            data_lines.append(stripped)
    return comment_lines, data_lines

comments_a, data_lines_a = split_comments_and_data(raw_lines_a)
comments_b, data_lines_b = split_comments_and_data(raw_lines_b)

print(f"Dataset A: {len(comments_a)} comment/metadata lines, {len(data_lines_a)} non-comment lines.")
print(f"Dataset B: {len(comments_b)} comment/metadata lines, {len(data_lines_b)} non-comment lines.")

Dataset A: 18 comment/metadata lines, 677437 non-comment lines.
Dataset B: 22 comment/metadata lines, 631455 non-comment lines.


**Full metadata/comment header — Dataset A (AncestryDNA):**

In [35]:
print("\n".join(comments_a))

#AncestryDNA raw data download
#This file was generated by AncestryDNA at: 06/22/2026 12:38:16 UTC
#Data was collected using AncestryDNA array version: V2.0
#Data is formatted using AncestryDNA converter version: V1.0
#Below is a text version of your DNA file from Ancestry.com DNA, LLC.  THIS 
#INFORMATION IS FOR YOUR PERSONAL USE AND IS INTENDED FOR GENEALOGICAL RESEARCH 
#ONLY.  IT IS NOT INTENDED FOR MEDICAL, DIAGNOSTIC, OR HEALTH PURPOSES.  THE EXPORTED DATA IS 
#SUBJECT TO THE ANCESTRY TERMS AND CONDITIONS, BUT PLEASE BE AWARE THAT THE 
#DOWNLOADED DATA WILL NO LONGER BE PROTECTED BY OUR SECURITY MEASURES.
#WHEN YOU DOWNLOAD YOUR RAW DNA DATA, YOU ASSUME ALL RISK OF STORING, 
#SECURING AND PROTECTING YOUR DATA.  FOR MORE INFORMATION, SEE ANCESTRYDNA FAQS. 
#
#Genetic data is provided below as five TAB delimited columns.  Each line 
#corresponds to a SNP.  Column one provides the SNP identifier (rsID where 
#possible).  Columns two and three contain the chromosome and basepair posi

**Full metadata/comment header — Dataset B (23andMe):**

In [36]:
print("\n".join(comments_b))

# ANONYMIZED RAW GENOTYPE DATA — FOR EDUCATIONAL USE ONLY
#
# Source platform : 23andMe v5 chip (direct-to-consumer SNP array)
# Reference build : GRCh37 / hg19 (NCBI Annotation Release 104)
# Orientation     : genotypes reported on the plus (+) strand of the reference
# Identifiers     : donor name, file_id, signature, timestamp and personalized
#                   account URL have been REMOVED for this teaching dataset.
#
# Format: tab-separated. One line per SNP.
#   column 1 = rsid        (dbSNP reference SNP id, or platform-internal 'i' id)
#   column 2 = chromosome  (1-22, X, Y, MT)
#   column 3 = position    (base-pair coordinate on GRCh37)
#   column 4 = genotype    (alleles on + strand; single allele for haploid
#                           male X/Y and mitochondrial calls; 'II'/'DD'/'DI'
#                           denote insertion/deletion calls; '--' = no-call)
#
# NOTE ON PRIVACY: a genotype profile is intrinsically personal and can in
# principle be re-identified. Removing

## 3. Structural Inspection

This section detects — rather than assumes — the file format, delimiter, encoding, and column structure of each dataset.

### 3.1 File Format Detection

In [37]:
def detect_file_format(path):
    """Report basic OS-level file characteristics without interpreting content."""
    size_bytes = os.path.getsize(path)
    with open(path, "rb") as f:
        first_bytes = f.read(4)
    return {
        "path": path,
        "size_bytes": size_bytes,
        "first_bytes_hex": first_bytes.hex(),
        "extension": os.path.splitext(path)[1],
    }

for label, path in DATASET_PATHS.items():
    info = detect_file_format(path)
    print(label)
    for k, v in info.items():
        print(f"  {k}: {v}")
    print()

Dataset A (AncestryDNA)
  path: F:\SRH Munich\2nd sem\Gemonics Analytics\PhenoPred\data\raw\AncestryDNA.txt
  size_bytes: 18320431
  first_bytes_hex: 23416e63
  extension: .txt

Dataset B (23andMe, Build 37)
  path: F:\SRH Munich\2nd sem\Gemonics Analytics\PhenoPred\data\raw\anonymous_genome_v5_build37.txt
  size_bytes: 16002658
  first_bytes_hex: 2320414e
  extension: .txt



### 3.2 Encoding Detection

Encoding is detected heuristically from the raw bytes. This is an observational check only — the file is not re-saved or re-encoded.

In [38]:
def detect_encoding_heuristic(path, sample_size=200_000):
    """
    Lightweight, dependency-free encoding heuristic:
    attempt strict decoding under a small set of common candidate encodings
    and report which succeed, plus whether a BOM is present.
    """
    with open(path, "rb") as f:
        raw = f.read(sample_size)

    bom_map = {
        b"\xef\xbb\xbf": "UTF-8 BOM",
        b"\xff\xfe": "UTF-16 LE BOM",
        b"\xfe\xff": "UTF-16 BE BOM",
    }
    detected_bom = next((name for sig, name in bom_map.items() if raw.startswith(sig)), None)

    candidates = ["ascii", "utf-8", "utf-8-sig", "latin-1"]
    decodable = {}
    for enc in candidates:
        try:
            raw.decode(enc)
            decodable[enc] = True
        except (UnicodeDecodeError, LookupError):
            decodable[enc] = False

    return {"bom_detected": detected_bom, "decodable_as": decodable}

for label, path in DATASET_PATHS.items():
    result = detect_encoding_heuristic(path)
    print(f"{label}:")
    print(f"  BOM detected     : {result['bom_detected']}")
    print(f"  Decodable as     : {result['decodable_as']}")
    print()

Dataset A (AncestryDNA):
  BOM detected     : None
  Decodable as     : {'ascii': True, 'utf-8': True, 'utf-8-sig': True, 'latin-1': True}

Dataset B (23andMe, Build 37):
  BOM detected     : None
  Decodable as     : {'ascii': False, 'utf-8': True, 'utf-8-sig': True, 'latin-1': True}



### 3.3 Line Ending Detection

Consumer genomics exports frequently differ in line-ending convention (CRLF vs LF) depending on the exporting platform. This is checked explicitly since it affects raw line parsing.

In [39]:
def detect_line_endings(path, sample_size=200_000):
    with open(path, "rb") as f:
        raw = f.read(sample_size)
    return {
        "CRLF (\\r\\n) count": raw.count(b"\r\n"),
        "lone LF (\\n) count": raw.count(b"\n") - raw.count(b"\r\n"),
        "lone CR (\\r) count": raw.count(b"\r") - raw.count(b"\r\n"),
    }

for label, path in DATASET_PATHS.items():
    print(label, "->", detect_line_endings(path))

Dataset A (AncestryDNA) -> {'CRLF (\\r\\n) count': 7668, 'lone LF (\\n) count': 0, 'lone CR (\\r) count': 0}
Dataset B (23andMe, Build 37) -> {'CRLF (\\r\\n) count': 0, 'lone LF (\\n) count': 8142, 'lone CR (\\r) count': 0}


### 3.4 Delimiter Detection

Delimiter is detected using `csv.Sniffer` on a sample of non-comment lines, as well as by direct character-frequency inspection, without assuming tab-separation in advance.

In [40]:
def detect_delimiter(data_lines, sample_lines=20):
    sample_text = "\n".join(data_lines[:sample_lines])
    result = {}
    try:
        dialect = csv.Sniffer().sniff(sample_text)
        result["csv_sniffer_delimiter"] = repr(dialect.delimiter)
    except csv.Error as e:
        result["csv_sniffer_delimiter"] = f"detection failed: {e}"

    # Direct character frequency check across candidate delimiters
    candidates = ["\t", ",", ";", "|", " "]
    freq = {}
    for cand in candidates:
        counts = [line.count(cand) for line in data_lines[:sample_lines] if line]
        freq[repr(cand)] = counts
    result["candidate_char_counts_first_lines"] = freq
    return result

print("Dataset A delimiter detection:")
delim_info_a = detect_delimiter(data_lines_a)
print(f"  csv.Sniffer result: {delim_info_a['csv_sniffer_delimiter']}")
print(f"  Candidate char counts (first lines): {delim_info_a['candidate_char_counts_first_lines']}")

print("\nDataset B delimiter detection:")
delim_info_b = detect_delimiter(data_lines_b)
print(f"  csv.Sniffer result: {delim_info_b['csv_sniffer_delimiter']}")
print(f"  Candidate char counts (first lines): {delim_info_b['candidate_char_counts_first_lines']}")

Dataset A delimiter detection:
  csv.Sniffer result: '\t'
  Candidate char counts (first lines): {"'\\t'": [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], "','": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "';'": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "'|'": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "' '": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

Dataset B delimiter detection:
  csv.Sniffer result: '\t'
  Candidate char counts (first lines): {"'\\t'": [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], "','": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "';'": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "'|'": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "' '": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


### 3.5 Column / Header Detection

The first non-comment line of each file is inspected to determine whether it is a column header row or already a data row, and to enumerate detected column names/positions.

In [41]:
DETECTED_DELIMITER = "\t"  # confirmed above via Sniffer and character-frequency analysis for both files

def inspect_header(data_lines, delimiter):
    first_line = data_lines[0]
    fields = first_line.split(delimiter)
    looks_like_header = not all(f.strip('#').strip().isdigit() or f.strip() == '' for f in fields)
    return first_line, fields, looks_like_header

first_a, fields_a, header_like_a = inspect_header(data_lines_a, DETECTED_DELIMITER)
first_b, fields_b, header_like_b = inspect_header(data_lines_b, DETECTED_DELIMITER)

print("Dataset A — first non-comment line:")
print(f"  raw     : {first_a!r}")
print(f"  fields  : {fields_a}")
print(f"  n_fields: {len(fields_a)}")
print()
print("Dataset B — first non-comment line:")
print(f"  raw     : {first_b!r}")
print(f"  fields  : {fields_b}")
print(f"  n_fields: {len(fields_b)}")

Dataset A — first non-comment line:
  raw     : 'rsid\tchromosome\tposition\tallele1\tallele2'
  fields  : ['rsid', 'chromosome', 'position', 'allele1', 'allele2']
  n_fields: 5

Dataset B — first non-comment line:
  raw     : 'rs548049170\t1\t69869\tTT'
  fields  : ['rs548049170', '1', '69869', 'TT']
  n_fields: 4


**Note:** Dataset B's column header appears as a *commented* line (`# rsid\tchromosome\t...`) rather than a plain header row, unlike Dataset A which has an uncommented `rsid\tchromosome\t...` header line. This is recorded as a structural difference for later pipeline design — it is not resolved or unified here.

In [42]:
# Locate the commented header line in Dataset B explicitly, without modifying the file.
# A genuine header comment is a delimiter-separated line whose FIRST token
# (after stripping the leading '#') is exactly 'rsid' -- this distinguishes it
# from prose comment lines that merely mention 'rsid' in passing (e.g.
# "column 1 = rsid (dbSNP reference SNP id...)").
b_header_comment_candidates = [
    c for c in comments_b
    if c.lstrip('#').strip().split(DETECTED_DELIMITER)[0].strip().lower() == 'rsid'
]
print("Dataset B commented line(s) identified as the header row (first token == 'rsid'):")
for c in b_header_comment_candidates:
    print(f"  {c!r}")

a_header_candidates = [l for l in data_lines_a[:1] if 'rsid' in l.lower()]
print("\nDataset A uncommented line(s) containing 'rsid':")
for c in a_header_candidates:
    print(f"  {c!r}")

Dataset B commented line(s) identified as the header row (first token == 'rsid'):
  '# rsid\tchromosome\tposition\tgenotype'

Dataset A uncommented line(s) containing 'rsid':
  'rsid\tchromosome\tposition\tallele1\tallele2'


## 4. Dataset Profiling

Each dataset is profiled independently. No cross-dataset comparison is performed in this section or elsewhere in this notebook.

In [43]:
def parse_data_rows(data_lines, delimiter, skip_header_if_present=True, header_keyword='rsid'):
    """
    Split each non-comment data line into fields, using the detected delimiter.
    If the first data line looks like a column header (contains the header
    keyword), it is returned separately as `header_row` and excluded from
    `rows`. No values are altered, cast, or dropped — only split.
    """
    header_row = None
    start_idx = 0
    if data_lines and header_keyword in data_lines[0].lower():
        header_row = data_lines[0].split(delimiter)
        start_idx = 1
    rows = [line.split(delimiter) for line in data_lines[start_idx:]]
    return header_row, rows

header_row_a, rows_a = parse_data_rows(data_lines_a, DETECTED_DELIMITER)
header_row_b, rows_b = parse_data_rows(data_lines_b, DETECTED_DELIMITER)

print(f"Dataset A header row detected: {header_row_a}")
print(f"Dataset A data rows (excl. header/comments): {len(rows_a):,}")
print()
print(f"Dataset B header row detected: {header_row_b}")
print(f"Dataset B data rows (excl. commented header/comments): {len(rows_b):,}")

Dataset A header row detected: ['rsid', 'chromosome', 'position', 'allele1', 'allele2']
Dataset A data rows (excl. header/comments): 677,436

Dataset B header row detected: None
Dataset B data rows (excl. commented header/comments): 631,455


### 4.1 Column Names

In [44]:
# Dataset A has an explicit uncommented header row.
COLUMNS_A = header_row_a if header_row_a else None

# Dataset B's header is only present as a comment; we surface it for reporting
# WITHOUT injecting it as if it were an uncommented data header (no modification
# of the file's actual structure — this is purely descriptive).
COLUMNS_B_FROM_COMMENT = None
if b_header_comment_candidates:
    # strip a single leading '#' and surrounding whitespace only for display purposes
    raw_comment_header = b_header_comment_candidates[0].lstrip('#').strip()
    COLUMNS_B_FROM_COMMENT = raw_comment_header.split(DETECTED_DELIMITER)

print(f"Dataset A columns (from uncommented header row): {COLUMNS_A}")
print(f"Dataset B columns (as documented in commented header line): {COLUMNS_B_FROM_COMMENT}")
print(f"Dataset B: no uncommented header row present in the data lines (confirmed above).")

Dataset A columns (from uncommented header row): ['rsid', 'chromosome', 'position', 'allele1', 'allele2']
Dataset B columns (as documented in commented header line): ['rsid', 'chromosome', 'position', 'genotype']
Dataset B: no uncommented header row present in the data lines (confirmed above).


### 4.2 Row and Column Counts

In [45]:
def column_count_distribution(rows):
    return collections.Counter(len(r) for r in rows)

col_dist_a = column_count_distribution(rows_a)
col_dist_b = column_count_distribution(rows_b)

print(f"Dataset A: {len(rows_a):,} data rows")
print(f"Dataset A: observed column-count distribution across rows: {dict(col_dist_a)}")
print()
print(f"Dataset B: {len(rows_b):,} data rows")
print(f"Dataset B: observed column-count distribution across rows: {dict(col_dist_b)}")

Dataset A: 677,436 data rows
Dataset A: observed column-count distribution across rows: {5: 677436}

Dataset B: 631,455 data rows
Dataset B: observed column-count distribution across rows: {4: 631455}


### 4.3 First 5 and Last 5 Rows (raw, unmodified)

In [46]:
print("Dataset A — header row:", header_row_a)
print("Dataset A — first 5 data rows:")
for r in rows_a[:5]:
    print(" ", r)
print("Dataset A — last 5 data rows:")
for r in rows_a[-5:]:
    print(" ", r)

Dataset A — header row: ['rsid', 'chromosome', 'position', 'allele1', 'allele2']
Dataset A — first 5 data rows:
  ['rs3131972', '1', '752721', 'A', 'G']
  ['rs114525117', '1', '759036', 'G', 'G']
  ['rs4040617', '1', '779322', 'A', 'G']
  ['rs141175086', '1', '780397', 'C', 'C']
  ['rs115093905', '1', '787173', 'G', 'G']
Dataset A — last 5 data rows:
  ['rs41534744', '26', '16129', 'A', 'A']
  ['rs41419246', '26', '16145', 'G', 'G']
  ['rs41466049', '26', '16162', 'A', 'A']
  ['rs41355449', '26', '16327', 'C', 'C']
  ['rs869031877', '26', '16391', 'G', 'G']


In [47]:
print("Dataset B — documented columns (from comment):", COLUMNS_B_FROM_COMMENT)
print("Dataset B — first 5 data rows:")
for r in rows_b[:5]:
    print(" ", r)
print("Dataset B — last 5 data rows:")
for r in rows_b[-5:]:
    print(" ", r)

Dataset B — documented columns (from comment): ['rsid', 'chromosome', 'position', 'genotype']
Dataset B — first 5 data rows:
  ['rs548049170', '1', '69869', 'TT']
  ['rs9283150', '1', '565508', 'AA']
  ['rs116587930', '1', '727841', 'AG']
  ['rs3131972', '1', '752721', 'AG']
  ['rs12184325', '1', '754105', 'CC']
Dataset B — last 5 data rows:
  ['i4000693', 'MT', '16524', 'A']
  ['i704756', 'MT', '16524', 'A']
  ['i705255', 'MT', '16525', 'A']
  ['i4000757', 'MT', '16526', 'G']
  ['i701671', 'MT', '16526', 'G']


### 4.4 Inferred Data Types

Types are *inferred by observation only* (e.g. "looks numeric", "looks alphabetic") for reporting purposes. No casting, coercion, or schema is imposed on the underlying data.

In [48]:
def infer_column_types(rows, n_columns, sample_size=5000):
    """
    For each column index, report the set of simple observed shapes
    (e.g. 'int-like', 'alpha', 'alphanumeric', 'empty') seen in a sample of rows.
    Only rows with exactly n_columns fields are included, to avoid misaligning
    ragged rows onto column indices (ragged rows are handled/reported in Section 5).
    """
    sample = [r for r in rows if len(r) == n_columns][:sample_size]
    per_col_shapes = [collections.Counter() for _ in range(n_columns)]
    for row in sample:
        for i, val in enumerate(row):
            if val == "":
                shape = "empty"
            elif val.isdigit():
                shape = "int-like"
            elif val.isalpha():
                shape = "alpha"
            elif val.isalnum():
                shape = "alphanumeric"
            else:
                shape = "other"
            per_col_shapes[i][shape] += 1
    return per_col_shapes

n_cols_a = max(col_dist_a, key=col_dist_a.get)
n_cols_b = max(col_dist_b, key=col_dist_b.get)

shapes_a = infer_column_types(rows_a, n_cols_a)
shapes_b = infer_column_types(rows_b, n_cols_b)

print("Dataset A — observed value shapes per column (modal column count = %d):" % n_cols_a)
for i, shp in enumerate(shapes_a):
    name = COLUMNS_A[i] if COLUMNS_A and i < len(COLUMNS_A) else f"col_{i}"
    print(f"  [{i}] {name}: {dict(shp)}")

print("\nDataset B — observed value shapes per column (modal column count = %d):" % n_cols_b)
for i, shp in enumerate(shapes_b):
    name = COLUMNS_B_FROM_COMMENT[i] if COLUMNS_B_FROM_COMMENT and i < len(COLUMNS_B_FROM_COMMENT) else f"col_{i}"
    print(f"  [{i}] {name}: {dict(shp)}")

Dataset A — observed value shapes per column (modal column count = 5):
  [0] rsid: {'alphanumeric': 5000}
  [1] chromosome: {'int-like': 5000}
  [2] position: {'int-like': 5000}
  [3] allele1: {'alpha': 4998, 'int-like': 2}
  [4] allele2: {'alpha': 4998, 'int-like': 2}

Dataset B — observed value shapes per column (modal column count = 4):
  [0] rsid: {'alphanumeric': 5000}
  [1] chromosome: {'int-like': 5000}
  [2] position: {'int-like': 5000}
  [3] genotype: {'alpha': 4928, 'other': 72}


## 5. Data Quality Checks (Observation Only)

All checks below are read-only observations. Nothing is removed, replaced, imputed, or corrected.

### 5.1 Duplicate Header Rows

In [49]:
def find_duplicate_header_rows(rows, header_row):
    """Find data rows that are identical repeats of the header row (e.g. header re-printed mid-file)."""
    if header_row is None:
        return []
    return [i for i, r in enumerate(rows) if r == header_row]

dup_headers_a = find_duplicate_header_rows(rows_a, header_row_a)
print(f"Dataset A: {len(dup_headers_a)} duplicate header row(s) found within data rows.")
if dup_headers_a:
    print(f"  Row indices (0-based, within data rows): {dup_headers_a[:20]}{'...' if len(dup_headers_a) > 20 else ''}")

# Dataset B has no uncommented header row to compare against; we instead check
# whether the documented header line reappears verbatim as a non-comment data line.
dup_headers_b = [i for i, r in enumerate(rows_b) if COLUMNS_B_FROM_COMMENT and r == COLUMNS_B_FROM_COMMENT]
print(f"\nDataset B: {len(dup_headers_b)} data row(s) matching the documented header found.")

Dataset A: 0 duplicate header row(s) found within data rows.

Dataset B: 0 data row(s) matching the documented header found.


### 5.2 Malformed Rows (Ragged Column Counts)

In [50]:
def find_malformed_rows(rows, expected_n_cols):
    return [(i, len(r), r) for i, r in enumerate(rows) if len(r) != expected_n_cols]

malformed_a = find_malformed_rows(rows_a, n_cols_a)
malformed_b = find_malformed_rows(rows_b, n_cols_b)

print(f"Dataset A: expected {n_cols_a} columns/row; {len(malformed_a)} malformed row(s) found.")
for i, n, r in malformed_a[:10]:
    print(f"  row {i}: {n} fields -> {r}")

print(f"\nDataset B: expected {n_cols_b} columns/row; {len(malformed_b)} malformed row(s) found.")
for i, n, r in malformed_b[:10]:
    print(f"  row {i}: {n} fields -> {r}")

Dataset A: expected 5 columns/row; 0 malformed row(s) found.

Dataset B: expected 4 columns/row; 0 malformed row(s) found.


### 5.3 Missing Value Encodings (Exactly as They Appear)

We scan the genotype/allele columns for tokens that plausibly represent 'missing' or 'no-call' data, reporting them **verbatim** — no assumption is made about which token(s) 'count' as missing beyond what is documented in the file's own metadata header.

In [51]:
def scan_value_frequencies(rows, col_index, top_n=20):
    counter = collections.Counter(r[col_index] for r in rows if len(r) > col_index)
    return counter.most_common(top_n)

# Dataset A: allele1 (col 3) and allele2 (col 4), 0-indexed
print("Dataset A — most common raw values in 'allele1' column:")
for val, cnt in scan_value_frequencies(rows_a, 3):
    print(f"  {val!r}: {cnt:,}")

print("\nDataset A — most common raw values in 'allele2' column:")
for val, cnt in scan_value_frequencies(rows_a, 4):
    print(f"  {val!r}: {cnt:,}")

Dataset A — most common raw values in 'allele1' column:
  'A': 204,102
  'T': 203,524
  'G': 130,860
  'C': 129,619
  'I': 6,419
  'D': 2,410
  '0': 502

Dataset A — most common raw values in 'allele2' column:
  'G': 228,066
  'C': 226,075
  'A': 107,203
  'T': 106,761
  'I': 6,427
  'D': 2,402
  '0': 502


In [52]:
# Dataset B: single 'genotype' column (col 3), 0-indexed, per its documented format
print("Dataset B — most common raw values in 'genotype' column:")
for val, cnt in scan_value_frequencies(rows_b, 3):
    print(f"  {val!r}: {cnt:,}")

Dataset B — most common raw values in 'genotype' column:
  'CC': 142,558
  'GG': 142,163
  'AA': 104,577
  'TT': 104,349
  'CT': 40,673
  'AG': 40,209
  '--': 10,093
  'AC': 9,849
  'GT': 9,760
  'C': 5,651
  'T': 5,462
  'A': 5,384
  'G': 5,340
  'II': 3,242
  'DD': 1,244
  'CG': 346
  'AT': 225
  'I': 197
  'D': 90
  'DI': 43


In [53]:
# Explicit count of the no-call token documented in Dataset B's own header ('--')
# and any values containing '-' more broadly, reported exactly as they appear.
nocall_b = sum(1 for r in rows_b if len(r) > 3 and r[3] == '--')
dash_containing_b = collections.Counter(r[3] for r in rows_b if len(r) > 3 and '-' in r[3])
print(f"Dataset B: rows with genotype exactly '--' : {nocall_b:,}")
print(f"Dataset B: distinct '-'-containing genotype tokens observed: {dict(dash_containing_b)}")

empty_a1 = sum(1 for r in rows_a if len(r) > 3 and r[3] == '')
empty_a2 = sum(1 for r in rows_a if len(r) > 4 and r[4] == '')
print(f"\nDataset A: rows with empty string in 'allele1': {empty_a1:,}")
print(f"Dataset A: rows with empty string in 'allele2': {empty_a2:,}")

zero_a = sum(1 for r in rows_a if len(r) > 3 and (r[3] == '0' or (len(r) > 4 and r[4] == '0')))
print(f"Dataset A: rows containing literal '0' token in allele1 and/or allele2: {zero_a:,}")

Dataset B: rows with genotype exactly '--' : 10,093
Dataset B: distinct '-'-containing genotype tokens observed: {'--': 10093}

Dataset A: rows with empty string in 'allele1': 0
Dataset A: rows with empty string in 'allele2': 0
Dataset A: rows containing literal '0' token in allele1 and/or allele2: 502


### 5.4 Missing Value Counts (Overall, Per Column)

In [54]:
def count_empty_fields_per_column(rows, n_columns):
    counts = [0] * n_columns
    for r in rows:
        for i in range(min(len(r), n_columns)):
            if r[i] == '':
                counts[i] += 1
    return counts

empty_counts_a = count_empty_fields_per_column(rows_a, n_cols_a)
empty_counts_b = count_empty_fields_per_column(rows_b, n_cols_b)

print("Dataset A — empty-string field counts per column:")
for i, c in enumerate(empty_counts_a):
    name = COLUMNS_A[i] if COLUMNS_A and i < len(COLUMNS_A) else f"col_{i}"
    print(f"  [{i}] {name}: {c:,}")

print("\nDataset B — empty-string field counts per column:")
for i, c in enumerate(empty_counts_b):
    name = COLUMNS_B_FROM_COMMENT[i] if COLUMNS_B_FROM_COMMENT and i < len(COLUMNS_B_FROM_COMMENT) else f"col_{i}"
    print(f"  [{i}] {name}: {c:,}")

Dataset A — empty-string field counts per column:
  [0] rsid: 0
  [1] chromosome: 0
  [2] position: 0
  [3] allele1: 0
  [4] allele2: 0

Dataset B — empty-string field counts per column:
  [0] rsid: 0
  [1] chromosome: 0
  [2] position: 0
  [3] genotype: 0


### 5.5 Duplicated RSIDs

In [55]:
def find_duplicate_rsids(rows, rsid_col=0):
    counter = collections.Counter(r[rsid_col] for r in rows if len(r) > rsid_col)
    return {rsid: cnt for rsid, cnt in counter.items() if cnt > 1}

dup_rsids_a = find_duplicate_rsids(rows_a)
dup_rsids_b = find_duplicate_rsids(rows_b)

print(f"Dataset A: {len(dup_rsids_a):,} distinct RSIDs appear more than once.")
print(f"  Example (first 10): {list(dup_rsids_a.items())[:10]}")

print(f"\nDataset B: {len(dup_rsids_b):,} distinct RSIDs appear more than once.")
print(f"  Example (first 10): {list(dup_rsids_b.items())[:10]}")

Dataset A: 0 distinct RSIDs appear more than once.
  Example (first 10): []

Dataset B: 0 distinct RSIDs appear more than once.
  Example (first 10): []


### 5.6 Duplicated Chromosome–Position Pairs

In [56]:
def find_duplicate_chr_pos(rows, chr_col=1, pos_col=2):
    counter = collections.Counter(
        (r[chr_col], r[pos_col]) for r in rows if len(r) > max(chr_col, pos_col)
    )
    return {k: v for k, v in counter.items() if v > 1}

dup_chrpos_a = find_duplicate_chr_pos(rows_a)
dup_chrpos_b = find_duplicate_chr_pos(rows_b)

print(f"Dataset A: {len(dup_chrpos_a):,} distinct (chromosome, position) pairs appear more than once.")
print(f"  Example (first 10): {list(dup_chrpos_a.items())[:10]}")

print(f"\nDataset B: {len(dup_chrpos_b):,} distinct (chromosome, position) pairs appear more than once.")
print(f"  Example (first 10): {list(dup_chrpos_b.items())[:10]}")

Dataset A: 614 distinct (chromosome, position) pairs appear more than once.
  Example (first 10): [(('1', '12027042'), 2), (('1', '17371286'), 2), (('1', '78407816'), 2), (('1', '94517225'), 2), (('1', '156105885'), 2), (('1', '170688894'), 2), (('1', '197070521'), 2), (('1', '197094290'), 2), (('1', '201333464'), 2), (('1', '218578520'), 2)]

Dataset B: 2,401 distinct (chromosome, position) pairs appear more than once.
  Example (first 10): [(('1', '2526746'), 2), (('1', '11009679'), 2), (('1', '19992513'), 2), (('1', '20020994'), 2), (('1', '21795388'), 2), (('1', '21902361'), 2), (('1', '21940555'), 2), (('1', '24421474'), 2), (('1', '24706292'), 2), (('1', '26664968'), 2)]


## 6. Genomic Structure Analysis (Observation Only)

This section describes the genomic notation and encoding conventions used by each file, strictly as observed. It does **not** attempt to normalize, remap, or reconcile these conventions between the two datasets.

### 6.1 Unique Chromosome Labels

In [57]:
def unique_chromosome_labels(rows, chr_col=1):
    return collections.Counter(r[chr_col] for r in rows if len(r) > chr_col)

chr_labels_a = unique_chromosome_labels(rows_a)
chr_labels_b = unique_chromosome_labels(rows_b)

def sort_key(label):
    return (0, int(label)) if label.isdigit() else (1, label)

print("Dataset A — unique chromosome labels and row counts:")
for label in sorted(chr_labels_a, key=sort_key):
    print(f"  {label!r}: {chr_labels_a[label]:,}")

print("\nDataset B — unique chromosome labels and row counts:")
for label in sorted(chr_labels_b, key=sort_key):
    print(f"  {label!r}: {chr_labels_b[label]:,}")

Dataset A — unique chromosome labels and row counts:
  '1': 50,554
  '2': 54,847
  '3': 43,310
  '4': 36,843
  '5': 38,796
  '6': 43,301
  '7': 34,621
  '8': 32,960
  '9': 29,616
  '10': 32,763
  '11': 32,527
  '12': 31,283
  '13': 24,919
  '14': 21,126
  '15': 21,344
  '16': 23,350
  '17': 22,891
  '18': 18,986
  '19': 16,969
  '20': 18,072
  '21': 10,167
  '22': 10,985
  '23': 25,242
  '24': 1,665
  '25': 36
  '26': 263

Dataset B — unique chromosome labels and row counts:
  '1': 48,902
  '2': 51,309
  '3': 42,578
  '4': 39,055
  '5': 36,707
  '6': 43,660
  '7': 34,005
  '8': 31,409
  '9': 26,135
  '10': 30,237
  '11': 30,668
  '12': 29,128
  '13': 21,936
  '14': 19,766
  '15': 18,799
  '16': 20,194
  '17': 19,199
  '18': 17,541
  '19': 14,677
  '20': 14,645
  '21': 8,490
  '22': 8,805
  'MT': 4,154
  'X': 16,119
  'Y': 3,337


### 6.2 Chromosome Notation System (Descriptive)

**Dataset A (AncestryDNA):** chromosome labels observed are purely numeric strings (`'1'`–`'22'`) plus higher numeric codes (e.g. `'23'`, `'24'`, `'25'`, `'26'`) that AncestryDNA's own convention typically reserves for X, Y, pseudoautosomal, and mitochondrial regions — but this file does **not** itself spell these out as `'X'`/`'Y'`/`'MT'`; it uses AncestryDNA's internal numeric coding scheme, as printed. No mapping from these numeric codes to `X`/`Y`/`MT` is performed in this notebook.

**Dataset B (23andMe):** chromosome labels observed include numeric strings (`'1'`–`'22'`) **and** the literal alphabetic tokens `'X'`, `'Y'`, `'MT'`, per the file's own documented header comment.

**Structural observation:** the two files use *different* labeling conventions for sex chromosomes and mitochondrial DNA (numeric-only vs. numeric+alphabetic). This is recorded as a structural fact for future pipeline design; no reconciliation, remapping, or 'correction' is attempted here.

### 6.3 Genotype Encoding Format (Descriptive)

In [58]:
# Dataset A: two separate allele columns (allele1, allele2)
allele_len_dist_a1 = collections.Counter(len(r[3]) for r in rows_a if len(r) > 3)
allele_len_dist_a2 = collections.Counter(len(r[4]) for r in rows_a if len(r) > 4)
print("Dataset A — allele1 string-length distribution:", dict(allele_len_dist_a1))
print("Dataset A — allele2 string-length distribution:", dict(allele_len_dist_a2))

# Dataset B: single combined genotype column
genotype_len_dist_b = collections.Counter(len(r[3]) for r in rows_b if len(r) > 3)
print("\nDataset B — genotype string-length distribution:", dict(genotype_len_dist_b))

Dataset A — allele1 string-length distribution: {1: 677436}
Dataset A — allele2 string-length distribution: {1: 677436}

Dataset B — genotype string-length distribution: {2: 609331, 1: 22124}


**Descriptive summary of encoding formats, as documented and observed:**

- **Dataset A (AncestryDNA):** genotype is split across **two columns** (`allele1`, `allele2`), each a single character in the sampled data (e.g. `A`, `G`, `C`, `T`), per the file's own header description ("forward (+) strand ... two alleles").
- **Dataset B (23andMe):** genotype is encoded in a **single column** as a string. Per the file's own header comment, this string is normally 2 characters for diploid autosomal calls (e.g. `AG`), but may be a **single character** for haploid calls (male X/Y, mitochondrial), or the tokens `II`/`DD`/`DI` for insertion/deletion calls, or `--` for no-calls. The length distribution above shows the range of lengths actually present.
- **Structural observation:** the two files represent genotype using fundamentally different column layouts (2-column vs. 1-column encoding) and, per Dataset B's documentation, Dataset B additionally encodes haploid calls and indel calls in ways that Dataset A's fixed two-column layout does not natively represent. This is a structural difference to be handled in future pipeline design; it is not resolved here.

### 6.4 Formatting Inconsistencies Observed (Haploid vs. Diploid, etc.)

In [59]:
# Dataset B: rows on X/Y/MT with single-character genotype strings (haploid-style calls)
haploid_like_b = [r for r in rows_b if len(r) > 3 and r[1] in ('X', 'Y', 'MT') and len(r[3]) == 1]
diploid_on_sexchr_b = [r for r in rows_b if len(r) > 3 and r[1] in ('X', 'Y', 'MT') and len(r[3]) == 2]

print(f"Dataset B: {len(haploid_like_b):,} rows on X/Y/MT with a single-character genotype (haploid-style).")
print(f"Dataset B: {len(diploid_on_sexchr_b):,} rows on X/Y/MT with a two-character genotype (diploid-style).")
print("Example haploid-style rows:")
for r in haploid_like_b[:5]:
    print(" ", r)
if diploid_on_sexchr_b:
    print("Example two-character rows on X/Y/MT:")
    for r in diploid_on_sexchr_b[:5]:
        print(" ", r)

Dataset B: 22,124 rows on X/Y/MT with a single-character genotype (haploid-style).
Dataset B: 1,486 rows on X/Y/MT with a two-character genotype (diploid-style).
Example haploid-style rows:
  ['i709476', 'X', '2700163', 'G']
  ['rs5939320', 'X', '2700202', 'A']
  ['i720421', 'X', '2701004', 'T']
  ['i720422', 'X', '2701185', 'T']
  ['rs113564299', 'X', '2701356', 'I']
Example two-character rows on X/Y/MT:
  ['i720207', 'X', '169786', 'GG']
  ['rs6423165', 'X', '169805', 'GG']
  ['i720209', 'X', '170193', 'GG']
  ['i720212', 'X', '172611', 'CC']
  ['i720216', 'X', '176827', 'CC']


In [60]:
# Dataset B: indel-style tokens as documented ('II', 'DD', 'DI')
indel_tokens_b = {tok: cnt for tok, cnt in collections.Counter(r[3] for r in rows_b if len(r) > 3).items()
                  if tok in ('II', 'DD', 'DI', 'ID')}
print(f"Dataset B: documented indel-style genotype tokens observed: {indel_tokens_b}")

# Dataset A: check whether allele1/allele2 are ever unequal length or contain non-single-character values
nonstandard_a1 = collections.Counter(r[3] for r in rows_a if len(r) > 3 and len(r[3]) != 1)
nonstandard_a2 = collections.Counter(r[4] for r in rows_a if len(r) > 4 and len(r[4]) != 1)
print(f"\nDataset A: allele1 values with length != 1: {dict(nonstandard_a1)}")
print(f"Dataset A: allele2 values with length != 1: {dict(nonstandard_a2)}")

Dataset B: documented indel-style genotype tokens observed: {'II': 3242, 'DD': 1244, 'DI': 43}

Dataset A: allele1 values with length != 1: {}
Dataset A: allele2 values with length != 1: {}


## 7. Summary

This section consolidates the structural observations above. It is descriptive only and intentionally makes **no** recommendations about cleaning, normalization, strand correction, or biological interpretation — those are pipeline-design decisions for a later, separate phase of the project.

### 7.1 Observed Structural Issues

- **Dataset A (AncestryDNA)**
  - Metadata occupies 18 leading comment lines (`#`), followed by an **uncommented** header row (`rsid\tchromosome\tposition\tallele1\tallele2`).
  - Genotype is split across two columns (`allele1`, `allele2`) rather than a single combined genotype column.
  - Chromosome labels are purely numeric strings, including codes beyond `'22'` (observed up to `'26'` in this file) whose biological meaning is not spelled out in-file.
  - Missing-value/no-call conventions were checked empirically (Section 5.3) rather than assumed — see printed results above for the exact tokens observed in the allele columns.
  - Duplicate-header, malformed-row, duplicate-RSID, and duplicate chromosome-position counts are reported in Section 5 and should be read directly from that section's output rather than restated here, to avoid drift between this summary and the executed results.

- **Dataset B (23andMe, Build 37)**
  - Metadata occupies 22 leading comment lines, and — notably — the **column header itself is only present as a commented line** (`# rsid\tchromosome\tposition\tgenotype`), with **no uncommented header row** in the data.
  - Genotype is encoded in a **single** column, whose string length varies (1 character for haploid/X/Y/MT calls per the file's own documentation, 2 characters for diploid calls, and special tokens `II`/`DD`/`DI` for indels, plus `--` for no-calls).
  - Chromosome labels include both numeric strings and the alphabetic tokens `X`, `Y`, `MT`.
  - RSID values include both standard dbSNP `rs...` identifiers and platform-internal `i...` identifiers, per the file's own header comment — this distinction was observed in the raw RSID values but not filtered or separated in this notebook.
  - Duplicate-header (commented-header-recurrence), malformed-row, duplicate-RSID, and duplicate chromosome-position counts are reported in Section 5 and should be read directly from that section's output.

### 7.2 Assumptions That CANNOT Be Made Yet

- It cannot yet be assumed that the two files' chromosome notations are equivalent or directly mappable (numeric-only vs. numeric+alphabetic).
- It cannot yet be assumed that AncestryDNA's higher numeric chromosome codes (e.g. `'25'`, `'26'`) correspond to any particular one of X, Y, pseudoautosomal region, or MT without an authoritative, documented mapping from AncestryDNA — none is asserted in this file's own header.
- It cannot yet be assumed that a given RSID refers to the *same physical genomic position* across both files (no cross-dataset comparison has been performed in this notebook).
- It cannot yet be assumed that both files share the same strand orientation convention beyond what each file's own header claims ("forward (+) strand" in both cases, as documented) — no strand verification or correction has been performed.
- It cannot yet be assumed that missing/no-call tokens are handled equivalently between the two files, since Dataset A and Dataset B use different column layouts and (per Section 5.3) potentially different literal tokens for missing data.
- It cannot yet be assumed that duplicate RSIDs or duplicate chromosome-position pairs (Sections 5.5–5.6) represent errors, multi-allelic probes, re-genotyped probes, or something else — no such judgement is made here.

### 7.3 Questions for Future Pipeline Design

1. How should the differing chromosome notations (numeric-only in Dataset A vs. numeric+alphabetic in Dataset B) be reconciled, and based on what authoritative reference?
2. How should Dataset A's non-standard numeric chromosome codes (beyond `'22'`) be interpreted, and what source should be used to confirm their meaning?
3. How should the two different genotype encodings (2-column allele pairs vs. 1-column genotype strings, including haploid/indel/no-call variants) be unified into a common representation, if at all?
4. What is the appropriate policy for handling missing/no-call values (observed in Section 5.3) — exclusion, explicit missingness flags, or another approach — and should it differ by column type (SNP vs. indel vs. sex-chromosome calls)?
5. How should duplicate RSIDs and duplicate chromosome-position pairs be investigated and handled — are they probe redesigns, multi-allelic sites, or data artifacts?
6. What strand-verification procedure (not performed here) would be required before any downstream comparison between the two datasets, given both claim forward-strand orientation but this has not been independently verified?
7. Should platform-internal identifiers (e.g. Dataset B's `i...` IDs) be treated differently from standard dbSNP `rs...` IDs in later processing?
8. What documentation or external reference is needed to confirm array version / build compatibility claims (AncestryDNA V2.0; 23andMe v5 chip, GRCh37) before any position-based operations are attempted?

---

*End of exploratory data analysis. No cleaning, transformation, comparison, or biological inference has been performed. All findings above are structural/descriptive and intended solely to inform the design of a future, separate preprocessing pipeline.*